# Anchor 主干训练划分证据

## tl;dr

找到与冻结权重对应的原运行 sidecar，并交叉核验 25,199 个样本。全部 8,192 个表达标准化参数精确一致，两个保存的预测字段在预定容差内复现。它补充原主干来源证据，不证明整个系统训练独立或生物学有效。

## Context & Methods

### Key Assumptions

原 sidecar 是归档证据，不是 checkpoint 内置划分。通过权重 SHA、模型类 AST、训练参数和全部保存预测交叉核对其对应关系。数值重放不是临床预测准确率。样本级原始证据保留在 SSD，不出现在此 notebook。

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
root = Path.cwd()
result = json.loads((root/'RESULT.json').read_text())
assert hashlib.sha256((root/'PROTOCOL.json').read_bytes()).hexdigest() == result['protocol_sha256']
assert result['sidecar_samples'] == 25199
assert result['split_disagreements_vs_previous_runtime'] == 0
assert result['missing_from_previous_runtime_split'] == 0
assert result['original_source_files_unchanged']
assert result['live_runtime_and_CPTAC_pool_unchanged']
print('原始材料、划分对应和未改动检查通过。')

原始材料、划分对应和未改动检查通过。


## Data

样本划分与固定 CPTAC 候选的既有病例关联：

In [2]:
assert sum(result['archived_split_counts'].values()) == 25199
assert result['CPTAC_candidate_anchor_linkage_counts'] == {'not_linked_by_existing_crosswalk':499, 'test':443}
pd.DataFrame.from_dict(result['archived_split_counts'], orient='index', columns=['样本数'])

,样本数
train,13107
val,8205
test,3887


## Results

这里比较的是数值记录一致性，不是与疾病或功能真值的比较。

In [3]:
for field in ['expr_mean','expr_std']:
    assert result['standardization_checks'][field]['exact_values'] == 4096
    assert result['standardization_checks'][field]['max_abs_error'] == 0
checks = pd.DataFrame(result['prediction_replay_checks']).T
assert checks['all_within_tolerance'].all()
assert (checks['within_tolerance_records'] == 25199).all()
checks[['records','max_abs_error','tolerance_abs','within_tolerance_records']]

,records,max_abs_error,tolerance_abs,within_tolerance_records
pred_age,25199,0.000011,0.001,25199
tumor_probability,25199,0.0,0.00001,25199


## Takeaways

原 anchor 主干 sidecar 已恢复并得到交叉支持。checkpoint 本身仍不含划分；后续 heads/priors/规则和患者级独立性未由本检查解决。499 个未关联 CPTAC 病例仍不能称为严格未暴露病例。原稿、正在运行的推断、候选池和权重均未修改。